### Task 2 - Create the dim_customer SCD Type 2 table
- create table
- load the initial customer records
- Load staging data with customer changes
- Apply SCD Type 2: Step 1 — Close changed records
- Apply SCD Type 2: Step 2 — Insert new versions and new customers

In [0]:
%sql
    
    
-- create table
CREATE or REPLACE TABLE IDENTIFIER(:catalog_name || '.2_silver.dim_customer') (
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_id STRING NOT NULL,
  full_name STRING,
  email STRING,
  city STRING,
  segment STRING,
  account_type STRING,
  valid_from TIMESTAMP NOT NULL,
  valid_to TIMESTAMP NOT NULL,
  is_current BOOLEAN
)
USING DELTA
CLUSTER BY (customer_id)
TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- -- verify the table exists
-- SELECT 
--   CASE 
--     WHEN count(*) > 0 THEN 'Table is created successfully!' 
--     ELSE 'Table dim_customer did not get created!' 
--   END AS check_result
-- FROM IDENTIFIER(:catalog_name || '.information_schema.tables')
-- WHERE table_schema = '2_silver' 
--   AND table_name = 'dim_customer';

In [0]:
# load the initial customer records
from pyspark.sql.functions import lit, to_timestamp

catalog_name = dbutils.widgets.get("catalog_name")

initial_customers = [
    ("C-1001", "Emma Hartley",      "emma.hartley@northbank.com",     "London",     "Retail",   "Savings"),
    ("C-1002", "James Weston",      "james.weston@northbank.com",     "Manchester", "Retail",   "Current"),
    ("C-1003", "Sophia Chen",       "sophia.chen@northbank.com",      "Edinburgh",  "Premium",  "Savings"),
    ("C-1004", "Oliver Banks",      "oliver.banks@northbank.com",     "Birmingham", "Retail",   "Current"),
    ("C-1005", "Aisha Patel",       "aisha.patel@northbank.com",      "London",     "Business", "Business"),
    ("C-1006", "Liam Murray",       "liam.murray@northbank.com",      "Leeds",      "Retail",   "Savings"),
    ("C-1007", "Charlotte Wright",  "charlotte.wright@northbank.com", "Bristol",    "Premium",  "Current"),
    ("C-1008", "Noah Thompson",     "noah.thompson@northbank.com",    "Glasgow",    "Retail",   "Savings"),
]

df_initial = spark.createDataFrame(
    initial_customers,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)

df_initial = (
    df_initial
    .withColumn("valid_from",  to_timestamp(lit("2020-01-01 00:00:00")))
    .withColumn("valid_to",    to_timestamp(lit("9999-12-31 00:00:00")))
    .withColumn("is_current",  lit(True))
)


df_initial.write.mode("overwrite").saveAsTable(f"{catalog_name}.2_silver.dim_customer")

print("Initial data loaded.")
#display(df_initial)